# FMI 3.0 Import

Running Functional Mock-up Units as FastSim blocks, in both interface types.

## What is an FMU?

[FMI](https://fmi-standard.org/) is an open standard for exchanging simulation models between tools. An FMU is a ZIP archive holding compiled model code, a `modelDescription.xml` declaring its variables and capabilities, and optional resources.

FastSim implements **FMI 3.0 natively in Rust** — it loads the shared library, parses the description and drives the `fmi3*` entry points itself. There is no Python FMI library in the loop, so an FMU is as cheap to step as any other block.

## Model Exchange or Co-Simulation?

The two interface types split the work differently:

| | Model Exchange | Co-Simulation |
|---|---|---|
| who integrates | FastSim's solver | the FMU's own solver |
| the FMU provides | $\dot{x} = f(x, u, t)$ | a step of size $\Delta t$ |
| step size | chosen by FastSim, adaptive if the solver is | the communication grid |
| events | located by FastSim via the FMU's event indicators | handled inside, reported back |

Model Exchange puts the FMU under one consistent integrator with the rest of the model, which is what you want for a stiff or tightly coupled system. Co-Simulation is the choice when the FMU's internal solver is part of the model's identity.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Apply the FastSim docs matplotlib style
plt.style.use('../fastsim_docs.mplstyle')

import fastsim as fs
from fastsim import Simulation, Connection
from fastsim.blocks import Scope
from fastsim._fastsim import ModelExchangeFMU, CoSimulationFMU
from fastsim.solvers import RKCK54

print(f"fastsim {fs.__version__}")

## The Reference FMUs

These examples use the [Reference-FMUs](https://github.com/modelica/Reference-FMUs) published by the Modelica Association — the same archives every FMI tool is tested against. FastSim keeps them as test fixtures, so the paths below point into the repository.

In [ ]:
from pathlib import Path

FIXTURES = Path("../../../tests/fixtures/fmi").resolve()

def fmu(name):
    path = FIXTURES / f"{name}.fmu"
    if not path.exists():
        raise FileNotFoundError(f"reference FMU not found: {path}")
    return str(path)

for name in ("Dahlquist", "BouncingBall"):
    print(f"{name:<14} {Path(fmu(name)).stat().st_size / 1024:5.0f} KiB")

## Model Exchange: Dahlquist

The Dahlquist test equation is the standard scalar stiffness probe:

$$\dot{x} = -k\,x, \qquad x(0) = 1 \quad\Rightarrow\quad x(t) = e^{-kt}$$

The FMU supplies $\dot{x}$; FastSim's solver does the integrating.

In [ ]:
f = ModelExchangeFMU(fmu("Dahlquist"))

# RKCK54 is adaptive and this problem is easy, so it takes very long steps. A
# Scope records at step points unless told otherwise; `sampling_period` puts the
# recording on a uniform grid instead, independent of what the solver does.
sco = Scope(labels=["x"], sampling_period=1e-2)

sim = Simulation(
    blocks=[f, sco],
    connections=[Connection(f, sco)],
    Solver=RKCK54,
    dt=1e-2,
    log=False,
)
sim.run(5.0)

t, [x] = sco.read()
print(f"{len(t)} samples, x(5) = {x[-1]:.9f}")

## Verification

The analytic solution is $e^{-t}$ (the FMU's default $k = 1$). Nothing about this check involves the FMU or FastSim — it is the exact answer.

In [ ]:
err = np.max(np.abs(x - np.exp(-t)))
print(f"worst |x - exp(-t)| over the run: {err:.3e}")

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 5), sharex=True)
ax1.plot(t, x, label="FMU via FastSim")
ax1.plot(t, np.exp(-t), "--", label="analytic")
ax1.set_ylabel("x")
ax1.legend()
ax2.semilogy(t, np.abs(x - np.exp(-t)) + 1e-18)
ax2.set_xlabel("time [s]")
ax2.set_ylabel("|error|")
plt.show()

## Overriding Start Values

Variables declared in `modelDescription.xml` can be set before instantiation. Here the decay rate $k$ goes from 1 to 3, so the solution must become $e^{-3t}$.

In [ ]:
f3 = ModelExchangeFMU(fmu("Dahlquist"), start_values={"k": 3.0})
sco3 = Scope(labels=["x"], sampling_period=1e-2)
sim3 = Simulation([f3, sco3], [Connection(f3, sco3)], Solver=RKCK54, dt=1e-2, log=False)
sim3.run(5.0)

t3, [x3] = sco3.read()
print(f"k=3: worst |x - exp(-3t)| = {np.max(np.abs(x3 - np.exp(-3 * t3))):.3e}")

An unknown name is rejected rather than silently ignored:

In [ ]:
try:
    ModelExchangeFMU(fmu("Dahlquist"), start_values={"not_a_variable": 1.0})
except ValueError as e:
    print(f"ValueError: {e}")

## Events: the Bouncing Ball

The BouncingBall FMU exposes an event indicator for the floor contact. In Model Exchange, FastSim locates the crossing, lets the FMU resolve the event, and carries on — the same machinery its own `ZeroCrossing` events use.

Outputs are height $h$ and velocity $v$.

In [ ]:
ball = ModelExchangeFMU(fmu("BouncingBall"), tolerance=1e-10)
sco_b = Scope(labels=["h", "v"], sampling_period=1e-3)

sim_b = Simulation(
    blocks=[ball, sco_b],
    connections=[Connection(ball[0], sco_b[0]), Connection(ball[1], sco_b[1])],
    Solver=RKCK54,
    dt=1e-3,
    log=False,
)
sim_b.run(3.0)

t_b, [h, v] = sco_b.read()
print(f"{len(t_b)} samples")

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 5), sharex=True)
ax1.plot(t_b, h)
ax1.axhline(0.0, color="#7F7F7F", lw=1, ls=":")
ax1.set_ylabel("height [m]")
ax2.plot(t_b, v, color="#377eb8")
ax2.set_xlabel("time [s]")
ax2.set_ylabel("velocity [m/s]")
plt.show()

## Verification

Two properties the physics fixes, independent of either implementation:

1. the ball never passes through the floor,
2. each bounce scales the energy by $e^2$, so successive apex heights satisfy $h_{n+1}/h_n = e^2$ with the same $e$ every time.

Apex heights are the robust quantity here: the samples immediately around a bounce straddle a discontinuity, so a velocity ratio taken from them depends on where the grid happens to fall. The apex does not.

In [ ]:
print(f"deepest penetration below the floor: {min(h):.2e} m")

# apexes: local maxima of the height trace, away from the floor
interior = np.flatnonzero((h[1:-1] > h[:-2]) & (h[1:-1] >= h[2:]) & (h[1:-1] > 1e-3)) + 1
apex_t, apex_h = t_b[interior], h[interior]
print(f"apexes found: {len(apex_h)}")
for n, (tt, hh) in enumerate(zip(apex_t, apex_h), 1):
    print(f"  apex {n} at t = {tt:.4f} s   h = {hh:.6f} m")

ratios = apex_h[1:] / apex_h[:-1]
print(f"height ratios h[n+1]/h[n]: {np.array2string(ratios, precision=4)}")
print(f"implied restitution e = sqrt(ratio): {np.array2string(np.sqrt(ratios), precision=4)}")
print(f"spread in e across bounces: {np.ptp(np.sqrt(ratios)):.2e}")

## Co-Simulation

The same FMU, now integrating itself. FastSim advances it on a communication grid and reads the outputs back; the FMU reports events it handled internally.

In [ ]:
ball_cs = CoSimulationFMU(fmu("BouncingBall"), dt=1e-3)
sco_cs = Scope(labels=["h", "v"], sampling_period=1e-3)

sim_cs = Simulation(
    blocks=[ball_cs, sco_cs],
    connections=[Connection(ball_cs[0], sco_cs[0]), Connection(ball_cs[1], sco_cs[1])],
    dt=1e-3,
    log=False,
)
sim_cs.run(3.0)

t_cs, [h_cs, v_cs] = sco_cs.read()

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(t_b, h, label="Model Exchange")
ax.plot(t_cs, h_cs, "--", label="Co-Simulation")
ax.axhline(0.0, color="#7F7F7F", lw=1, ls=":")
ax.set_xlabel("time [s]")
ax.set_ylabel("height [m]")
ax.legend()
plt.show()

The two agree on the trajectory but not bit-for-bit, and they should not: Model Exchange puts the ball under FastSim's adaptive solver with the event located to `tolerance`, while Co-Simulation hands each step to the FMU's own integrator on a fixed grid. Before the first bounce both are integrating the same smooth free fall, and what separates them there is the reference FMU's own forward-Euler step: over a fall of duration $t$ that lags the exact parabola by $	frac{1}{2}g\,\Delta t\,t$, which is the number below. After the first bounce the bounce instants drift by a fraction of a communication step, and near a bounce a small shift in time is a large difference in height — so the whole-run figure is dominated by that, not by any disagreement about the physics.

In [ ]:
# Before the first bounce both are integrating the same smooth free fall.
# The bounce is where the velocity flips from falling to rising.
first_bounce = t_b[np.flatnonzero((v[:-1] < 0) & (v[1:] > 0))[0]]
pre = np.linspace(0, first_bounce * 0.95, 500)
gap_pre = np.max(np.abs(np.interp(pre, t_b, h) - np.interp(pre, t_cs, h_cs)))
print(f"before the first bounce (t < {pre[-1]:.3f} s): {gap_pre:.3e} m")

# The reference FMU integrates itself with forward Euler in Co-Simulation. Over
# free fall that lags the true parabola by 0.5*g*dt*t — which is the gap above.
print(f"  forward-Euler lag 0.5*g*dt*t at that t:  {0.5 * 9.81 * 1e-3 * pre[-1]:.3e} m")

# Afterwards the bounce instants drift apart, and near a bounce a small time
# shift is a large height difference.
grid = np.linspace(0, 3.0, 3001)
gap_all = np.max(np.abs(np.interp(grid, t_b, h) - np.interp(grid, t_cs, h_cs)))
print(f"over the whole run:                {gap_all:.3e} m")
print(f"communication step:                {1e-3:.0e} s")